# machine probe
Cores, memory and interpreter of the submit machine; nothing is played.


In [ ]:
import os
import sys
import time

T0 = time.time()
# the competition: a notebook runs at most 9 hours, internet off (the competition page, read 2026-10-30)
KAGGLE_WALL_S = 9 * 3600
# PLAY's whole-run margin (play_chain.sh: e4b_run.py ... 300); its end path measured 104-119 s (PLAY24 rounds 1-6)
MARGIN_S = 300
# rule A's slice (Isaiah 2026-10-23), at the values Isaiah set for Kaggle A 2026-10-30 21:28 ("like 300s-400 seconds")
SLICE_MIN_S, SLICE_MAX_S = 300, 400
# the learned lineage is required: this agent is born once and carries on from it (Isaiah 2026-10-30 21:28)
REQUIRE_LINEAGE = True
INPUT = os.environ.get("NB_INPUT", "/kaggle/input")
WORKING = os.environ.get("NB_WORKING", "/kaggle/working")
GATEWAY = os.environ.get("NB_GATEWAY", "http://gateway:8001")
WHEELS = os.path.join(INPUT, "competitions", "arc-prize-2026-arc-agi-3", "arc_agi_3_wheels")
N_GAMES = os.environ.get("NB_N_GAMES")      # local runs only; on Kaggle every listed game is played
RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
DEADLINE_AT = T0 + KAGGLE_WALL_S - MARGIN_S
print(f"T0 {T0:.1f} deadline_at {DEADLINE_AT:.1f} rerun {RERUN} slice {SLICE_MIN_S}/{SLICE_MAX_S} "
      f"input {INPUT} working {WORKING} gateway {GATEWAY} n_games {N_GAMES}")


In [ ]:
import platform
import shutil

mem = {}
if os.path.isfile("/proc/meminfo"):
    with open("/proc/meminfo") as fh:
        mem = {ln.split(":")[0]: ln.split(":")[1].strip() for ln in fh if ":" in ln}
cg = None
for p in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
    if os.path.isfile(p):
        with open(p) as fh:
            cg = (p, fh.read().strip())
        break
print(f"python {sys.version.split()[0]} {platform.platform()}")
print(f"nproc os.cpu_count {os.cpu_count()} sched_getaffinity "
      f"{len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else None}")
print(f"MemTotal {mem.get('MemTotal')} MemAvailable {mem.get('MemAvailable')} cgroup {cg}")
print(f"disk working free {shutil.disk_usage(WORKING if os.path.isdir(WORKING) else '.').free}")
# how the inputs were stored (an unpacked folder or a .zip): two levels, names and sizes
for d1 in sorted(os.listdir(INPUT)) if os.path.isdir(INPUT) else []:
    p1 = os.path.join(INPUT, d1)
    print("input", d1, "" if os.path.isdir(p1) else os.path.getsize(p1))
    for d2 in sorted(os.listdir(p1))[:40] if os.path.isdir(p1) else []:
        p2 = os.path.join(p1, d2)
        print("   ", d2, "<dir>" if os.path.isdir(p2) else os.path.getsize(p2))


In [ ]:
# Non-rerun mode: produce a dummy submission (the competition's sample, its import moved under the test: a rerun
# never needs pandas)
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd

    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet(os.path.join(WORKING, 'submission.parquet'), index=False)
